# EMCAD ClinicDB — Final 5-Run Testing, Aggregation, and Evidence (Robust)


In [1]:
!nvidia-smi
!python --version

Sat Oct  3 04:26:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path
import hashlib
import json

input_root=Path("/kaggle/input")
working=Path("/kaggle/working")
working.mkdir(exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

def select_unique(filename,label):
    paths=sorted(set(input_root.rglob(filename)))
    if not paths:
        raise FileNotFoundError(f"{label} not found: {filename}")

    if len(paths)==1:
        return paths[0]

    hashes={}
    for path in paths:
        hashes.setdefault(sha256(path),[]).append(path)

    if len(hashes)==1:
        print(f"{label}: {len(paths)} identical copies found; using first copy.")
        return paths[0]

    details={k:[str(p) for p in v] for k,v in hashes.items()}
    raise RuntimeError(f"{label}: conflicting copies found: {details}")

env_archive=select_unique("emcadenv.tar.gz","Notebook 1 environment")
repo_archive=select_unique("EMCAD_prepared.tar.gz","Notebook 1 EMCAD repository")
nb1_checksum=select_unique("EMCAD_NOTEBOOK1_SHA256SUMS.txt","Notebook 1 checksum")

if not (env_archive.parent==repo_archive.parent==nb1_checksum.parent):
    raise RuntimeError(
        "Notebook 1 artifacts were found in different input folders. "
        "Attach one consistent saved Notebook 1 output."
    )

run_archives={}
for run_number in range(1,6):
    run_archives[run_number]=select_unique(
        f"EMCAD_run{run_number}_training_output.tar.gz",
        f"Run {run_number}"
    )

manifest={
    "notebook1_root":str(env_archive.parent),
    "env_archive":str(env_archive),
    "repo_archive":str(repo_archive),
    "notebook1_checksum":str(nb1_checksum),
    "runs":{str(k):str(v) for k,v in run_archives.items()}
}

(working/"input_manifest.json").write_text(json.dumps(manifest,indent=2))

print("INPUTS VERIFIED")
print("Notebook 1:",env_archive.parent)
for run_number,path in run_archives.items():
    print(f"Run {run_number}:",path)


INPUTS VERIFIED
Notebook 1: /kaggle/input/notebooks/anirbanroysudipto/notebook-1-environment-preparation
Run 1: /kaggle/input/notebooks/anirbanroysudipto/split-training-run-1/EMCAD_run1_training_output.tar.gz
Run 2: /kaggle/input/notebooks/rinkyranisaha/split-training-run-2/EMCAD_run2_training_output.tar.gz
Run 3: /kaggle/input/notebooks/rinkyranisaha/split-training-run-3/EMCAD_run3_training_output.tar.gz
Run 4: /kaggle/input/notebooks/sajibbiswas42/split-training-run-4/EMCAD_run4_training_output.tar.gz
Run 5: /kaggle/input/notebooks/sajibbiswas42/split-training-run-5/EMCAD_run5_training_output.tar.gz


In [3]:
from pathlib import Path
import json
import shutil
import subprocess
import tarfile

working=Path("/kaggle/working")
manifest=json.loads((working/"input_manifest.json").read_text())

env_dir=working/"emcadenv"
repo_dir=working/"EMCAD"

for path in [env_dir,repo_dir,working/"run_inputs"]:
    if path.exists():
        shutil.rmtree(path)

env_dir.mkdir()
(working/"run_inputs").mkdir()

# Verify Notebook 1 output exactly as it was saved.
subprocess.run(
    ["sha256sum","-c",Path(manifest["notebook1_checksum"]).name],
    cwd=Path(manifest["notebook1_checksum"]).parent,
    check=True
)

with tarfile.open(manifest["env_archive"],"r:gz") as tf:
    tf.extractall(env_dir)

subprocess.run([str(env_dir/"bin"/"conda-unpack")],check=True)

with tarfile.open(manifest["repo_archive"],"r:gz") as tf:
    tf.extractall(working)

assert repo_dir.is_dir()

print("Notebook 1 environment and clean EMCAD repository restored.")


emcadenv.tar.gz: OK
EMCAD_prepared.tar.gz: OK
EMCAD_AUTHOR_COMMIT.txt: OK
ClinicDB_author_split_manifest.csv: OK
EMCAD_ENVIRONMENT.txt: OK
PVT_V2_B2_SHA256.txt: OK


/tmp/ipykernel_23/3991774610.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(env_dir)
/tmp/ipykernel_23/3991774610.py:33: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(working)


Notebook 1 environment and clean EMCAD repository restored.


In [4]:
import subprocess
from pathlib import Path

ENV=Path("/kaggle/working/emcadenv")
REPO=Path("/kaggle/working/EMCAD")
EXPECTED="26c9c31f731f749b62c5fe83f44376dac75f3aa8"

commit=subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],
    text=True
).strip()

assert commit==EXPECTED,(commit,EXPECTED)

subprocess.run(["git","-C",str(REPO),"diff","--exit-code"],check=True)
subprocess.run(["git","-C",str(REPO),"diff","--cached","--exit-code"],check=True)

status=subprocess.check_output(
    ["git","-C",str(REPO),"status","--porcelain","--untracked-files=no"],
    text=True
).strip()

assert status=="",status

subprocess.run(
    [str(ENV/"bin"/"python"),"-c",
     "import sys,torch,numpy,timm,ptflops;"
     "print('Python:',sys.version.split()[0]);"
     "print('PyTorch:',torch.__version__);"
     "print('CUDA build:',torch.version.cuda);"
     "print('CUDA available:',torch.cuda.is_available());"
     "print('GPU count:',torch.cuda.device_count());"
     "print('GPU 0:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None');"
     "print('NumPy:',numpy.__version__);"
     "print('timm:',timm.__version__);"
     "assert sys.version_info[:2]==(3,8);"
     "assert torch.__version__.startswith('1.11.0');"
     "assert torch.cuda.is_available()"],
    check=True
)

print("Author commit and clean source verified:",commit)


Python: 3.8.20
PyTorch: 1.11.0+cu113
CUDA build: 11.3
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
NumPy: 1.22.4
timm: 0.6.12
Author commit and clean source verified: 26c9c31f731f749b62c5fe83f44376dac75f3aa8


In [5]:
from pathlib import Path
import tarfile
import json
import hashlib
import shutil
import pandas as pd

working=Path("/kaggle/working")
repo=working/"EMCAD"
extract_root=working/"run_inputs"
manifest=json.loads((working/"input_manifest.json").read_text())
expected_commit="26c9c31f731f749b62c5fe83f44376dac75f3aa8"

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

# Clean model/log folders from the prepared repository before restoring our five runs.
for folder_name in ["model_pth","logs"]:
    folder=repo/folder_name
    folder.mkdir(exist_ok=True)
    for item in folder.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

rows=[]
run_ids={}

for run_number in range(1,6):
    archive=Path(manifest["runs"][str(run_number)])
    destination=extract_root/f"run{run_number}"
    destination.mkdir(parents=True,exist_ok=True)

    with tarfile.open(archive,"r:gz") as tf:
        tf.extractall(destination)

    metadata_paths=list(destination.rglob("run_metadata.json"))
    assert len(metadata_paths)==1,(run_number,metadata_paths)

    package=metadata_paths[0].parent
    metadata=json.loads(metadata_paths[0].read_text())

    expected={
        "run_number":run_number,
        "author_commit":expected_commit,
        "author_tracked_status":"",
        "dataset":"ClinicDB",
        "train_count":489,
        "val_count":61,
        "test_count":62,
        "encoder":"pvt_v2_b2",
        "decoder":"EMCAD",
        "kernel_sizes":[1,3,5],
        "lgag_kernel_size":3,
        "expansion_factor":2,
        "image_size":352,
        "batch_size":8,
        "epochs":200
    }

    for key,value in expected.items():
        assert metadata[key]==value,(run_number,key,metadata[key],value)

    assert abs(float(metadata["learning_rate"])-0.0005)<1e-12

    checksum_file=package/"SHA256SUMS.txt"
    assert checksum_file.is_file()

    for line in checksum_file.read_text().splitlines():
        checksum,relative=line.split(maxsplit=1)
        file_path=package/relative.strip()
        assert file_path.is_file(),file_path
        assert sha256(file_path)==checksum,(run_number,file_path)

    model_dirs=[p for p in (package/"model_pth").iterdir() if p.is_dir()]
    assert len(model_dirs)==1,(run_number,model_dirs)

    source_model_dir=model_dirs[0]
    run_id=source_model_dir.name

    assert f"_run{run_number}_" in run_id,run_id
    assert (source_model_dir/f"{run_id}-best.pth").is_file()
    assert (source_model_dir/f"{run_id}-last.pth").is_file()

    shutil.copytree(source_model_dir,repo/"model_pth"/run_id)

    train_log=package/"logs"/f"train_log_{run_id}.log"
    assert train_log.is_file()
    shutil.copy2(train_log,repo/"logs"/train_log.name)

    metadata["archive_sha256"]=sha256(archive)
    metadata["archive_input_path"]=str(archive)

    rows.append(metadata)
    run_ids[run_number]=run_id

metadata_df=pd.DataFrame(rows).sort_values("run_number").reset_index(drop=True)
metadata_df.to_csv(working/"five_run_training_metadata.csv",index=False)
(working/"run_ids.json").write_text(
    json.dumps({str(k):v for k,v in run_ids.items()},indent=2)
)

print("ALL FIVE TRAINING RUNS VERIFIED")
display(metadata_df[[
    "run_number","best_val_dice","test_dice_at_best_val",
    "author_total_train_time_seconds","wall_clock_seconds"
]])

for run_number,run_id in run_ids.items():
    print(run_number,run_id)


/tmp/ipykernel_23/2382708069.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(destination)


ALL FIVE TRAINING RUNS VERIFIED


,run_number,best_val_dice,test_dice_at_best_val,author_total_train_time_seconds,wall_clock_seconds
0,1,0.9217,0.9401,37533.87,38260
1,2,0.9058,0.9451,38403.51,39132
2,3,0.9209,0.9411,34621.86,35323
3,4,0.9115,0.9394,34030.84,34713
4,5,0.9199,0.9402,32966.86,33647


1 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008
2 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run2_t165126
3 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run3_t171839
4 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run4_t173922
5 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run5_t175741


In [6]:
from pathlib import Path
import subprocess

repo=Path("/kaggle/working/EMCAD")

expected_counts={"train":489,"val":61,"test":62}

for split,expected in expected_counts.items():
    image_dir=repo/"data/polyp/target/ClinicDB"/split/"images"
    mask_dir=repo/"data/polyp/target/ClinicDB"/split/"masks"

    images=list(image_dir.glob("*.png"))
    masks=list(mask_dir.glob("*.png"))

    print(split,"images=",len(images),"masks=",len(masks))
    assert len(images)==expected
    assert len(masks)==expected

model_dirs=[p for p in (repo/"model_pth").iterdir() if p.is_dir()]
assert len(model_dirs)==5

subprocess.run(["git","-C",str(repo),"diff","--exit-code"],check=True)
subprocess.run(["git","-C",str(repo),"diff","--cached","--exit-code"],check=True)

print("Prepared ClinicDB, five model directories, and clean author source verified.")


train images= 489 masks= 489
val images= 61 masks= 61
test images= 62 masks= 62
Prepared ClinicDB, five model directories, and clean author source verified.


## Official author testing — Runs 1–5

This runs the authors' unchanged `test_polyp.py`. The script writes each run into its own `predictions_polyp/<run_id>/...` directory, so post-hoc analysis uses the exact run ID instead of searching ambiguous snapshot folders.

In [7]:
from pathlib import Path
import json
import shutil
import subprocess

working=Path("/kaggle/working")
repo=working/"EMCAD"
env=working/"emcadenv"
run_ids=json.loads((working/"run_ids.json").read_text())

for path in [
    repo/"results_polyp",
    repo/"predictions_polyp"
]:
    if path.exists():
        shutil.rmtree(path)

summary_file=repo/"All_Runs_Summary_Polyp.xlsx"
if summary_file.exists():
    summary_file.unlink()

stdout_file=working/"author_test_stdout.log"
if stdout_file.exists():
    stdout_file.unlink()

with stdout_file.open("w") as log:
    for run_number in range(1,6):
        run_id=run_ids[str(run_number)]

        print("="*70)
        print("OFFICIAL TESTING RUN",run_number)
        print(run_id)
        print("="*70)

        proc=subprocess.Popen(
            [
                str(env/"bin"/"python"),
                "-u","-W","ignore",
                "test_polyp.py",
                "--run_id",run_id
            ],
            cwd=repo,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in proc.stdout:
            print(line,end="")
            log.write(line)
            log.flush()

        rc=proc.wait()
        if rc!=0:
            raise RuntimeError(f"Official test_polyp.py failed for Run {run_number} with code {rc}")

        pred_dir=repo/"predictions_polyp"/run_id/"ClinicDB"/"test"
        result_file=repo/"results_polyp"/f"Results_{run_id}_ClinicDB_test.xlsx"

        assert pred_dir.is_dir(),pred_dir
        assert len(list(pred_dir.glob("*.png")))==62
        assert result_file.is_file(),result_file

assert summary_file.is_file()

print("Official testing for all five runs completed.")


OFFICIAL TESTING RUN 1
ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515

Inference on test: 100%|██████████| 62/62 [00:03<00:00, 17.58it/s]
Evaluation complete. Summary appended to All_Runs_Summary_Polyp.xlsx
OFFICIAL TESTING RUN 2
ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run2_t165126
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515

Inference on test: 100%|██████████| 62/62 [00:03<00:00, 18.61it/s]
Evaluation complete. Summary appended to All_Runs_Summary_Polyp.xlsx
OFFICIAL TESTING RUN 3
ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run3_t171839
Model pvt_v2_b2 backbone:  created, param

In [8]:
from pathlib import Path
import pandas as pd
import json

working=Path("/kaggle/working")
repo=working/"EMCAD"
metadata=pd.read_csv(working/"five_run_training_metadata.csv")

summary=pd.read_excel(repo/"All_Runs_Summary_Polyp.xlsx")
run_to_number=dict(zip(metadata["run_id"],metadata["run_number"].astype(int)))

summary["run_number"]=summary["run_id"].map(run_to_number)
assert summary["run_number"].notna().all()
summary=summary.sort_values("run_number").reset_index(drop=True)

filtered=summary[
    (summary["dataset"].astype(str).str.lower()=="clinicdb") &
    (summary["split"].astype(str).str.lower()=="test")
].copy()

assert len(filtered)==5

metric_columns=["dice","iou","sensitivity","specificity","precision","HD95"]
for column in metric_columns:
    assert column in filtered.columns,column

aggregate_rows=[]
for column in metric_columns:
    values=pd.to_numeric(filtered[column],errors="raise")
    aggregate_rows.append({
        "metric":column,
        "mean":values.mean(),
        "std":values.std(ddof=1),
        "min":values.min(),
        "max":values.max()
    })

official_aggregate=pd.DataFrame(aggregate_rows)

published_dice=0.9521
replicated_dice=float(
    official_aggregate.loc[
        official_aggregate["metric"]=="dice","mean"
    ].iloc[0]
)

paper_reference=pd.DataFrame([{
    "model":"PVT-EMCAD-B2",
    "dataset":"ClinicDB",
    "published_dice_percent":95.21,
    "published_dice":published_dice,
    "replicated_5run_mean_dice":replicated_dice,
    "replicated_5run_mean_dice_percent":replicated_dice*100,
    "difference_percentage_points":(replicated_dice-published_dice)*100,
    "absolute_difference":abs(replicated_dice-published_dice)
}])

filtered.to_csv(working/"official_test_results.csv",index=False)
official_aggregate.to_csv(working/"official_test_results_mean_std.csv",index=False)
paper_reference.to_csv(working/"paper_reference_comparison.csv",index=False)

print("OFFICIAL AUTHOR TEST RESULTS")
display(filtered)

print("OFFICIAL FIVE-RUN MEAN ± STD")
display(official_aggregate)

print("PUBLISHED REFERENCE COMPARISON")
display(paper_reference)


OFFICIAL AUTHOR TEST RESULTS


,run_id,network,dataset,split,dice,iou,sensitivity,specificity,precision,HD95,run_number
0,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.940087,0.892741,0.957248,0.996134,0.929794,14.316587,1
1,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.945079,0.900617,0.956937,0.996450,0.938232,8.444777,2
2,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.941128,0.894039,0.954947,0.996408,0.933505,9.119384,3
3,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.939436,0.892459,0.960884,0.995734,0.925879,9.427935,4
4,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.940245,0.892479,0.952285,0.996247,0.935368,9.191661,5


OFFICIAL FIVE-RUN MEAN ± STD


,metric,mean,std,min,max
0,dice,0.941195,0.002253,0.939436,0.945079
1,iou,0.894467,0.003499,0.892459,0.900617
2,sensitivity,0.956460,0.003168,0.952285,0.960884
3,specificity,0.996195,0.000287,0.995734,0.996450
4,precision,0.932555,0.004828,0.925879,0.938232
5,HD95,10.100069,2.385253,8.444777,14.316587


PUBLISHED REFERENCE COMPARISON


,model,dataset,published_dice_percent,published_dice,replicated_5run_mean_dice,replicated_5run_mean_dice_percent,difference_percentage_points,absolute_difference
0,PVT-EMCAD-B2,ClinicDB,95.21,0.9521,0.941195,94.119497,-1.090503,0.010905


## Aggregate all 200-epoch training histories

In [9]:
from pathlib import Path
import pandas as pd
import re

working=Path("/kaggle/working")
repo=working/"EMCAD"
metadata=pd.read_csv(working/"five_run_training_metadata.csv")

pattern=re.compile(
    r"Epoch:\s*(\d+),\s*Dataset:\s*(test|val),\s*Dice:\s*([0-9.]+),\s*IoU:\s*([0-9.]+)",
    re.I
)

rows=[]

for _,meta in metadata.iterrows():
    run_number=int(meta["run_number"])
    run_id=str(meta["run_id"])
    log_path=repo/"logs"/f"train_log_{run_id}.log"

    assert log_path.is_file(),log_path

    for line in log_path.read_text(errors="ignore").splitlines():
        match=pattern.search(line)
        if match:
            rows.append({
                "run_number":run_number,
                "run_id":run_id,
                "epoch":int(match.group(1)),
                "split":match.group(2).lower(),
                "dice":float(match.group(3)),
                "iou":float(match.group(4))
            })

history=pd.DataFrame(rows)
assert not history.empty
assert history["run_number"].nunique()==5

for run_number in range(1,6):
    group=history[history["run_number"]==run_number]
    assert group[group["split"]=="val"]["epoch"].nunique()==200
    assert group[group["split"]=="test"]["epoch"].nunique()==200

best_rows=[]

for run_number,group in history.groupby("run_number"):
    val=group[group["split"]=="val"].sort_values("epoch")
    test=group[group["split"]=="test"].sort_values("epoch")

    best_index=val["dice"].idxmax()
    best_epoch=int(val.loc[best_index,"epoch"])
    test_row=test[test["epoch"]==best_epoch]

    assert len(test_row)==1

    best_rows.append({
        "run_number":int(run_number),
        "best_epoch":best_epoch,
        "best_val_dice_from_log":float(val.loc[best_index,"dice"]),
        "best_val_iou_from_log":float(val.loc[best_index,"iou"]),
        "test_dice_at_best_val_from_log":float(test_row.iloc[0]["dice"]),
        "test_iou_at_best_val_from_log":float(test_row.iloc[0]["iou"])
    })

best_epochs=pd.DataFrame(best_rows).sort_values("run_number")

history.to_csv(working/"epoch_history_all_runs.csv",index=False)
best_epochs.to_csv(working/"best_epoch_summary.csv",index=False)

display(best_epochs)


,run_number,best_epoch,best_val_dice_from_log,best_val_iou_from_log,test_dice_at_best_val_from_log,test_iou_at_best_val_from_log
0,1,115,0.9217,0.8652,0.9401,0.8927
1,2,144,0.9058,0.8462,0.9451,0.9006
2,3,156,0.9209,0.8667,0.9411,0.8940
3,4,153,0.9115,0.8525,0.9394,0.8925
4,5,169,0.9199,0.8656,0.9402,0.8925


## Post-hoc extra metrics

This block uses the **authors' own `get_loader`** to reconstruct the test targets exactly as `test_polyp.py` does. It reads each run's prediction directory directly by exact `run_id`, avoiding the previous ambiguous-directory error.

It also cross-checks post-hoc Dice/IoU against the authors' per-image Excel results before accepting the extra metrics.

In [10]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
REPO=/kaggle/working/EMCAD

cd "$REPO"

$ENV/bin/python - <<'PY'
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from utils.dataloader_polyp import get_loader

working=Path("/kaggle/working")
repo=Path("/kaggle/working/EMCAD")
metadata=pd.read_csv(working/"five_run_training_metadata.csv").sort_values("run_number")

image_root=repo/"data/polyp/target/ClinicDB/test/images"
gt_root=repo/"data/polyp/target/ClinicDB/test/masks"

loader=get_loader(
    image_root=str(image_root)+"/",
    gt_root=str(gt_root)+"/",
    batchsize=1,
    trainsize=352,
    shuffle=False,
    augmentation=False,
    split="test",
    color_image=True
)

# Build exactly the same binary GT geometry used by test_polyp.py.
targets={}

for pack in loader:
    images,gts,original_shapes,names=pack

    h_orig=int(original_shapes[0][0])
    w_orig=int(original_shapes[1][0])

    g=gts[0].unsqueeze(0).float()
    gt_resized=F.interpolate(
        g,
        size=(h_orig,w_orig),
        mode="nearest"
    ).squeeze()

    target=(gt_resized>=0.2).cpu().numpy().astype(bool)
    targets[names[0]]=target

assert len(targets)==62

eps=1e-8
smooth=1e-6

def calculate(pred,gt):
    pred=pred.astype(bool)
    gt=gt.astype(bool)

    tp=int(np.logical_and(pred,gt).sum())
    tn=int(np.logical_and(~pred,~gt).sum())
    fp=int(np.logical_and(pred,~gt).sum())
    fn=int(np.logical_and(~pred,gt).sum())

    sensitivity=tp/(tp+fn+eps)
    specificity=tn/(tn+fp+eps)
    precision=tp/(tp+fp+eps)

    dice=(2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou=(tp+smooth)/(tp+fp+fn+smooth)

    return {
        "tp":tp,
        "tn":tn,
        "fp":fp,
        "fn":fn,
        "dice":dice,
        "iou":iou,
        "sensitivity":sensitivity,
        "specificity":specificity,
        "precision":precision,
        "npv":tn/(tn+fn+eps),
        "accuracy":(tp+tn)/(tp+tn+fp+fn+eps),
        "balanced_accuracy":(sensitivity+specificity)/2,
        "f1":2*precision*sensitivity/(precision+sensitivity+eps),
        "fpr":fp/(fp+tn+eps),
        "fnr":fn/(fn+tp+eps),
        "mcc":(tp*tn-fp*fn)/np.sqrt(
            (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)+eps
        )
    }

rows=[]
validation_rows=[]

for _,meta in metadata.iterrows():
    run_number=int(meta["run_number"])
    run_id=str(meta["run_id"])

    pred_dir=repo/"predictions_polyp"/run_id/"ClinicDB"/"test"
    assert pred_dir.is_dir(),pred_dir

    pred_paths=sorted(pred_dir.glob("*.png"),key=lambda p:int(p.stem))
    assert len(pred_paths)==62,(run_number,len(pred_paths))

    official_file=repo/"results_polyp"/f"Results_{run_id}_ClinicDB_test.xlsx"
    assert official_file.is_file(),official_file

    official=pd.read_excel(official_file)
    official=official[official["Name"].astype(str)!="AVERAGE"].copy()
    assert len(official)==62

    official_by_name=official.set_index("Name")

    run_dice_diff=[]
    run_iou_diff=[]

    for pred_path in pred_paths:
        name=pred_path.name
        assert name in targets,name
        assert name in official_by_name.index,name

        pred_img=cv2.imread(str(pred_path),cv2.IMREAD_GRAYSCALE)
        assert pred_img is not None,pred_path

        gt=targets[name]
        assert pred_img.shape==gt.shape,(run_number,name,pred_img.shape,gt.shape)

        pred=pred_img>=128
        metrics=calculate(pred,gt)

        author_dice=float(official_by_name.loc[name,"Dice"])
        author_iou=float(official_by_name.loc[name,"IoU"])

        dice_diff=abs(metrics["dice"]-author_dice)
        iou_diff=abs(metrics["iou"]-author_iou)

        run_dice_diff.append(dice_diff)
        run_iou_diff.append(iou_diff)

        row={
            "run_number":run_number,
            "run_id":run_id,
            "image":name
        }
        row.update(metrics)
        rows.append(row)

    validation_rows.append({
        "run_number":run_number,
        "max_abs_dice_difference_vs_author":max(run_dice_diff),
        "max_abs_iou_difference_vs_author":max(run_iou_diff)
    })

per_image=pd.DataFrame(rows)
validation=pd.DataFrame(validation_rows)

assert len(per_image)==5*62
assert (validation["max_abs_dice_difference_vs_author"]<1e-5).all(),validation
assert (validation["max_abs_iou_difference_vs_author"]<1e-5).all(),validation

metric_cols=[
    "dice","iou","sensitivity","specificity","precision",
    "npv","accuracy","balanced_accuracy","f1","fpr","fnr","mcc"
]

macro_by_run=(
    per_image
    .groupby(["run_number","run_id"],as_index=False)[metric_cols]
    .mean()
)

pooled_rows=[]

for (run_number,run_id),group in per_image.groupby(["run_number","run_id"]):
    tp=int(group["tp"].sum())
    tn=int(group["tn"].sum())
    fp=int(group["fp"].sum())
    fn=int(group["fn"].sum())

    sensitivity=tp/(tp+fn+eps)
    specificity=tn/(tn+fp+eps)
    precision=tp/(tp+fp+eps)

    pooled_rows.append({
        "run_number":run_number,
        "run_id":run_id,
        "tp":tp,
        "tn":tn,
        "fp":fp,
        "fn":fn,
        "dice":(2*tp+smooth)/(2*tp+fp+fn+smooth),
        "iou":(tp+smooth)/(tp+fp+fn+smooth),
        "sensitivity":sensitivity,
        "specificity":specificity,
        "precision":precision,
        "npv":tn/(tn+fn+eps),
        "accuracy":(tp+tn)/(tp+tn+fp+fn+eps),
        "balanced_accuracy":(sensitivity+specificity)/2,
        "f1":2*precision*sensitivity/(precision+sensitivity+eps),
        "fpr":fp/(fp+tn+eps),
        "fnr":fn/(fn+tp+eps),
        "mcc":(tp*tn-fp*fn)/np.sqrt(
            (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)+eps
        )
    })

pooled_by_run=pd.DataFrame(pooled_rows)

extra_aggregate=pd.DataFrame([
    {
        "metric":metric,
        "mean":macro_by_run[metric].mean(),
        "std":macro_by_run[metric].std(ddof=1),
        "min":macro_by_run[metric].min(),
        "max":macro_by_run[metric].max()
    }
    for metric in metric_cols
])

per_image.to_csv(working/"extra_metrics_per_image.csv",index=False)
macro_by_run.to_csv(working/"extra_metrics_macro_by_run.csv",index=False)
pooled_by_run.to_csv(working/"extra_metrics_pooled_by_run.csv",index=False)
extra_aggregate.to_csv(working/"extra_metrics_5run_aggregate.csv",index=False)
validation.to_csv(working/"posthoc_validation_vs_author.csv",index=False)

print("POST-HOC VALIDATION AGAINST AUTHOR PER-IMAGE RESULTS")
print(validation.to_string(index=False))

print("\nPOST-HOC MACRO METRICS BY RUN")
print(macro_by_run.to_string(index=False))

print("\nPOST-HOC FIVE-RUN AGGREGATE")
print(extra_aggregate.to_string(index=False))
PY


POST-HOC VALIDATION AGAINST AUTHOR PER-IMAGE RESULTS
 run_number  max_abs_dice_difference_vs_author  max_abs_iou_difference_vs_author
          1                       2.914831e-08                      2.854378e-08
          2                       2.951616e-08                      2.985205e-08
          3                       2.928891e-08                      2.916795e-08
          4                       2.966690e-08                      2.942092e-08
          5                       2.902559e-08                      2.973017e-08

POST-HOC MACRO METRICS BY RUN
 run_number                                                                                                                              run_id     dice      iou  sensitivity  specificity  precision      npv  accuracy  balanced_accuracy       f1      fpr      fnr      mcc
          1 ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008 0.940087 0.89

## Architecture trace, kernel sizes, stage input/output sizes, FLOPs and parameters

This is read-only analysis. The first best checkpoint is loaded with the same `strict=False` behavior used by the authors' test script.

In [11]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
REPO=/kaggle/working/EMCAD

cd "$REPO"

$ENV/bin/python - <<'PY'
from pathlib import Path
import pandas as pd
import torch
from thop import profile

from lib.networks import EMCADNet

working=Path("/kaggle/working")
repo=Path("/kaggle/working/EMCAD")
metadata=pd.read_csv(working/"five_run_training_metadata.csv").sort_values("run_number")

run_id=str(metadata.iloc[0]["run_id"])
checkpoint=repo/"model_pth"/run_id/f"{run_id}-best.pth"
assert checkpoint.is_file()

model=EMCADNet(
    num_classes=1,
    kernel_sizes=[1,3,5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation="relu6",
    encoder="pvt_v2_b2",
    pretrain=False
).cuda()

state=torch.load(checkpoint,map_location="cuda")
load_info=model.load_state_dict(state,strict=False)
model.eval()

def shape_of(obj):
    if torch.is_tensor(obj):
        return list(obj.shape)
    if isinstance(obj,(list,tuple)):
        return [shape_of(x) for x in obj]
    if isinstance(obj,dict):
        return {k:shape_of(v) for k,v in obj.items()}
    return str(type(obj).__name__)

stage_rows=[]
stage_handles=[]

for name,module in model.named_children():
    def make_stage_hook(module_name):
        def hook(mod,inp,out):
            stage_rows.append({
                "stage":module_name,
                "type":mod.__class__.__name__,
                "input_size":str(shape_of(inp)),
                "output_size":str(shape_of(out)),
                "parameters_in_stage":sum(p.numel() for p in mod.parameters())
            })
        return hook

    stage_handles.append(module.register_forward_hook(make_stage_hook(name)))

layer_rows=[]
layer_handles=[]
counter={"n":0}

for name,module in model.named_modules():
    if name=="" or len(list(module.children()))>0:
        continue

    def make_layer_hook(module_name):
        def hook(mod,inp,out):
            counter["n"]+=1
            layer_rows.append({
                "call":counter["n"],
                "stage":module_name.split(".")[0],
                "module":module_name,
                "type":mod.__class__.__name__,
                "kernel_size":str(getattr(mod,"kernel_size","")),
                "stride":str(getattr(mod,"stride","")),
                "padding":str(getattr(mod,"padding","")),
                "dilation":str(getattr(mod,"dilation","")),
                "groups":str(getattr(mod,"groups","")),
                "input_size":str(shape_of(inp)),
                "output_size":str(shape_of(out)),
                "parameters_direct":sum(
                    p.numel() for p in mod.parameters(recurse=False)
                )
            })
        return hook

    layer_handles.append(module.register_forward_hook(make_layer_hook(name)))

x=torch.zeros(1,3,352,352,device="cuda")

with torch.no_grad():
    outputs=model(x)

for handle in stage_handles+layer_handles:
    handle.remove()

stage_df=pd.DataFrame(stage_rows)
layer_df=pd.DataFrame(layer_rows)

model_io=pd.DataFrame([{
    "input_size":str(list(x.shape)),
    "output_size":str(shape_of(outputs)),
    "number_of_outputs":len(outputs) if isinstance(outputs,(list,tuple)) else 1
}])

# Use THOP, matching the authors' own cal_params_flops implementation.
profile_input=torch.randn(1,3,352,352,device="cuda")
flops,params=profile(model,inputs=(profile_input,),verbose=False)
total_params=sum(p.numel() for p in model.parameters())

configuration=pd.DataFrame([{
    "dataset":"ClinicDB",
    "encoder":"pvt_v2_b2",
    "decoder":"EMCAD",
    "input_size":"352x352",
    "MSDC_kernel_sizes":"[1,3,5]",
    "LGAG_kernel_size":3,
    "expansion_factor":2,
    "depthwise_parallel":True,
    "aggregation":"add",
    "activation":"relu6",
    "batch_size":8,
    "epochs":200,
    "learning_rate":0.0005,
    "runs":5,
    "FLOPs_G":flops/1e9,
    "THOP_params_M":params/1e6,
    "total_parameters_M":total_params/1e6,
    "checkpoint_missing_keys":str(list(load_info.missing_keys)),
    "checkpoint_unexpected_keys":str(list(load_info.unexpected_keys))
}])

stage_df.to_csv(working/"architecture_stage_trace.csv",index=False)
layer_df.to_csv(working/"architecture_layer_trace.csv",index=False)
model_io.to_csv(working/"model_input_output.csv",index=False)
configuration.to_csv(working/"model_configuration_and_complexity.csv",index=False)

print("Model input:",list(x.shape))
print("Model output:",shape_of(outputs))
print("Top-level stages:",len(stage_df))
print("Leaf-module calls:",len(layer_df))
print("FLOPs (G):",flops/1e9)
print("THOP params (M):",params/1e6)
print("Total parameters (M):",total_params/1e6)
print("Missing keys:",list(load_info.missing_keys))
print("Unexpected keys:",list(load_info.unexpected_keys))
PY


Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
Model input: [1, 3, 352, 352]
Model output: [[1, 1, 352, 352], [1, 1, 352, 352], [1, 1, 352, 352], [1, 1, 352, 352]]
Top-level stages: 6
Leaf-module calls: 421
FLOPs (G): 10.579868042
THOP params (M): 26.764399
Total parameters (M): 26.764411
Missing keys: []
Unexpected keys: ['total_ops', 'total_params', 'backbone.total_ops', 'backbone.total_params', 'backbone.patch_embed1.total_ops', 'backbone.patch_embed1.total_params', 'backbone.patch_embed2.total_ops', 'backbone.patch_embed2.total_params', 'backbone.patch_embed3.total_ops', 'backbone.patch_embed3.total_params', 'backbone.patch_embed4.total_ops', 'backbone.patch_embed4.total_params', 'backbone.block1.total_ops', 'backbone.block1.total_params', 'backbone.block1.0.total_ops', 'backbone.block1.0.total_params', 'backbone.block1.0.attn.total_ops', 'backbone.block1.0.attn.total_params', 'backbone.block1.0.drop_path.total_ops', 'b

## Build final workbook and evidence package

In [12]:
from pathlib import Path
import pandas as pd
import shutil

working=Path("/kaggle/working")
repo=working/"EMCAD"
final_dir=working/"EMCAD_FINAL_RESULTS"

if final_dir.exists():
    shutil.rmtree(final_dir)
final_dir.mkdir()

official=pd.read_csv(working/"official_test_results.csv")
official_aggregate=pd.read_csv(working/"official_test_results_mean_std.csv")
paper_reference=pd.read_csv(working/"paper_reference_comparison.csv")
metadata=pd.read_csv(working/"five_run_training_metadata.csv")
history=pd.read_csv(working/"epoch_history_all_runs.csv")
best_epochs=pd.read_csv(working/"best_epoch_summary.csv")
extra_per_image=pd.read_csv(working/"extra_metrics_per_image.csv")
extra_macro=pd.read_csv(working/"extra_metrics_macro_by_run.csv")
extra_pooled=pd.read_csv(working/"extra_metrics_pooled_by_run.csv")
extra_aggregate=pd.read_csv(working/"extra_metrics_5run_aggregate.csv")
posthoc_validation=pd.read_csv(working/"posthoc_validation_vs_author.csv")
stage_trace=pd.read_csv(working/"architecture_stage_trace.csv")
layer_trace=pd.read_csv(working/"architecture_layer_trace.csv")
model_io=pd.read_csv(working/"model_input_output.csv")
configuration=pd.read_csv(working/"model_configuration_and_complexity.csv")

# Combine all five official per-image result sheets.
official_per_image_parts=[]

for _,meta in metadata.sort_values("run_number").iterrows():
    run_number=int(meta["run_number"])
    run_id=str(meta["run_id"])

    path=repo/"results_polyp"/f"Results_{run_id}_ClinicDB_test.xlsx"
    df=pd.read_excel(path)
    df=df[df["Name"].astype(str)!="AVERAGE"].copy()
    df.insert(0,"run_id",run_id)
    df.insert(0,"run_number",run_number)
    official_per_image_parts.append(df)

official_per_image=pd.concat(official_per_image_parts,ignore_index=True)
assert len(official_per_image)==310

official_per_image.to_csv(final_dir/"official_per_image_results.csv",index=False)

workbook=final_dir/"EMCAD_ClinicDB_5Run_Replication_Results.xlsx"

with pd.ExcelWriter(workbook,engine="openpyxl") as writer:
    official.to_excel(writer,sheet_name="official_test_results",index=False)
    official_aggregate.to_excel(writer,sheet_name="official_mean_std",index=False)
    official_per_image.to_excel(writer,sheet_name="official_per_image",index=False)
    paper_reference.to_excel(writer,sheet_name="paper_reference",index=False)
    metadata.to_excel(writer,sheet_name="training_metadata",index=False)
    best_epochs.to_excel(writer,sheet_name="best_epoch",index=False)
    history.to_excel(writer,sheet_name="epoch_history",index=False)
    extra_macro.to_excel(writer,sheet_name="extra_macro_by_run",index=False)
    extra_pooled.to_excel(writer,sheet_name="extra_pooled_by_run",index=False)
    extra_aggregate.to_excel(writer,sheet_name="extra_5run_summary",index=False)
    extra_per_image.to_excel(writer,sheet_name="extra_per_image",index=False)
    posthoc_validation.to_excel(writer,sheet_name="posthoc_validation",index=False)
    configuration.to_excel(writer,sheet_name="configuration",index=False)
    model_io.to_excel(writer,sheet_name="model_io",index=False)
    stage_trace.to_excel(writer,sheet_name="stage_trace",index=False)
    layer_trace.to_excel(writer,sheet_name="layer_trace",index=False)

csv_names=[
    "official_test_results.csv",
    "official_test_results_mean_std.csv",
    "paper_reference_comparison.csv",
    "five_run_training_metadata.csv",
    "best_epoch_summary.csv",
    "epoch_history_all_runs.csv",
    "extra_metrics_per_image.csv",
    "extra_metrics_macro_by_run.csv",
    "extra_metrics_pooled_by_run.csv",
    "extra_metrics_5run_aggregate.csv",
    "posthoc_validation_vs_author.csv",
    "architecture_stage_trace.csv",
    "architecture_layer_trace.csv",
    "model_input_output.csv",
    "model_configuration_and_complexity.csv"
]

for name in csv_names:
    shutil.copy2(working/name,final_dir/name)

shutil.copy2(
    repo/"All_Runs_Summary_Polyp.xlsx",
    final_dir/"AUTHOR_All_Runs_Summary_Polyp.xlsx"
)

shutil.copy2(
    working/"author_test_stdout.log",
    final_dir/"author_test_stdout.log"
)

shutil.copytree(
    repo/"results_polyp",
    final_dir/"AUTHOR_results_polyp"
)

shutil.copytree(
    repo/"predictions_polyp",
    final_dir/"AUTHOR_predictions_polyp"
)

# Training-run provenance/evidence without duplicating checkpoints.
evidence_root=final_dir/"training_run_evidence"
evidence_root.mkdir()

for run_number in range(1,6):
    metadata_paths=list(
        (working/"run_inputs"/f"run{run_number}").rglob("run_metadata.json")
    )
    assert len(metadata_paths)==1

    package=metadata_paths[0].parent
    target=evidence_root/f"run{run_number}"
    target.mkdir()

    for filename in [
        "run_metadata.json",
        "SHA256SUMS.txt",
        "author_train_stdout.log"
    ]:
        shutil.copy2(package/filename,target/filename)

    if (package/"evidence").exists():
        shutil.copytree(package/"evidence",target/"evidence")

print("Final workbook:",workbook)
print("Final result directory:",final_dir)


Final workbook: /kaggle/working/EMCAD_FINAL_RESULTS/EMCAD_ClinicDB_5Run_Replication_Results.xlsx
Final result directory: /kaggle/working/EMCAD_FINAL_RESULTS


In [13]:
from pathlib import Path
import hashlib
import shutil
import subprocess
import tarfile

working=Path("/kaggle/working")
repo=working/"EMCAD"
final_dir=working/"EMCAD_FINAL_RESULTS"
checkpoint_dir=working/"EMCAD_BEST_CHECKPOINTS"

# Final source-integrity verification.
subprocess.run(["git","-C",str(repo),"diff","--exit-code"],check=True)
subprocess.run(["git","-C",str(repo),"diff","--cached","--exit-code"],check=True)

tracked_status=subprocess.check_output(
    ["git","-C",str(repo),"status","--porcelain","--untracked-files=no"],
    text=True
).strip()

assert tracked_status=="",tracked_status

if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)
checkpoint_dir.mkdir()

model_dirs=sorted([p for p in (repo/"model_pth").iterdir() if p.is_dir()])
assert len(model_dirs)==5

for run_dir in model_dirs:
    run_id=run_dir.name
    checkpoint=run_dir/f"{run_id}-best.pth"
    assert checkpoint.is_file()
    shutil.copy2(checkpoint,checkpoint_dir/checkpoint.name)

assert len(list(checkpoint_dir.glob("*-best.pth")))==5

result_zip=working/"EMCAD_ClinicDB_5Run_Final_Results.zip"
checkpoint_tar=working/"EMCAD_ClinicDB_5Run_Best_Checkpoints.tar.gz"
checksum_file=working/"EMCAD_ClinicDB_5Run_Final_SHA256SUMS.txt"

for path in [result_zip,checkpoint_tar,checksum_file]:
    if path.exists():
        path.unlink()

shutil.make_archive(
    str(result_zip.with_suffix("")),
    "zip",
    root_dir=final_dir
)

with tarfile.open(checkpoint_tar,"w:gz") as tf:
    tf.add(checkpoint_dir,arcname=checkpoint_dir.name)

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

workbook=final_dir/"EMCAD_ClinicDB_5Run_Replication_Results.xlsx"

with checksum_file.open("w") as f:
    for path in [result_zip,checkpoint_tar,workbook]:
        f.write(f"{sha256(path)}  {path.name}\n")

print("Final verification: tracked author source files are unchanged.")
print()
for path in [result_zip,checkpoint_tar,workbook,checksum_file]:
    print(path.name,f"{path.stat().st_size/1024**2:.2f} MB")


Final verification: tracked author source files are unchanged.

EMCAD_ClinicDB_5Run_Final_Results.zip 0.73 MB
EMCAD_ClinicDB_5Run_Best_Checkpoints.tar.gz 474.29 MB
EMCAD_ClinicDB_5Run_Replication_Results.xlsx 0.17 MB
EMCAD_ClinicDB_5Run_Final_SHA256SUMS.txt 0.00 MB


In [14]:
from pathlib import Path
import pandas as pd

working=Path("/kaggle/working")
final_dir=working/"EMCAD_FINAL_RESULTS"

official=pd.read_csv(final_dir/"official_test_results.csv")
official_aggregate=pd.read_csv(final_dir/"official_test_results_mean_std.csv")
reference=pd.read_csv(final_dir/"paper_reference_comparison.csv")
best=pd.read_csv(final_dir/"best_epoch_summary.csv")
extra=pd.read_csv(final_dir/"extra_metrics_macro_by_run.csv")
validation=pd.read_csv(final_dir/"posthoc_validation_vs_author.csv")
configuration=pd.read_csv(final_dir/"model_configuration_and_complexity.csv")

print("OFFICIAL AUTHOR TEST RESULTS")
display(official)

print("OFFICIAL FIVE-RUN MEAN ± STD")
display(official_aggregate)

print("BEST EPOCHS FROM AUTHOR TRAINING LOGS")
display(best)

print("PUBLISHED REFERENCE COMPARISON")
display(reference)

print("POST-HOC EXTRA METRICS")
display(extra)

print("POST-HOC VALIDATION AGAINST AUTHOR DICE/IoU")
display(validation)

print("MODEL CONFIGURATION / COMPLEXITY")
display(configuration)

print()
print("FINAL OUTPUTS")

for name in [
    "EMCAD_ClinicDB_5Run_Final_Results.zip",
    "EMCAD_ClinicDB_5Run_Best_Checkpoints.tar.gz",
    "EMCAD_ClinicDB_5Run_Final_SHA256SUMS.txt"
]:
    path=working/name
    print(name,f"{path.stat().st_size/1024**2:.2f} MB")

workbook=final_dir/"EMCAD_ClinicDB_5Run_Replication_Results.xlsx"
print(workbook.name,f"{workbook.stat().st_size/1024**2:.2f} MB")

print()
print("FINAL NOTEBOOK COMPLETE")


OFFICIAL AUTHOR TEST RESULTS


,run_id,network,dataset,split,dice,iou,sensitivity,specificity,precision,HD95,run_number
0,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.940087,0.892741,0.957248,0.996134,0.929794,14.316587,1
1,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.945079,0.900617,0.956937,0.996450,0.938232,8.444777,2
2,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.941128,0.894039,0.954947,0.996408,0.933505,9.119384,3
3,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.939436,0.892459,0.960884,0.995734,0.925879,9.427935,4
4,ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw...,EMCADNet,ClinicDB,test,0.940245,0.892479,0.952285,0.996247,0.935368,9.191661,5


OFFICIAL FIVE-RUN MEAN ± STD


,metric,mean,std,min,max
0,dice,0.941195,0.002253,0.939436,0.945079
1,iou,0.894467,0.003499,0.892459,0.900617
2,sensitivity,0.956460,0.003168,0.952285,0.960884
3,specificity,0.996195,0.000287,0.995734,0.996450
4,precision,0.932555,0.004828,0.925879,0.938232
5,HD95,10.100069,2.385253,8.444777,14.316587


BEST EPOCHS FROM AUTHOR TRAINING LOGS


...

POST-HOC VALIDATION AGAINST AUTHOR DICE/IoU


,run_number,max_abs_dice_difference_vs_author,max_abs_iou_difference_vs_author
0,1,2.914831e-08,2.854378e-08
1,2,2.951616e-08,2.985205e-08
2,3,2.928891e-08,2.916795e-08
3,4,2.966690e-08,2.942092e-08
4,5,2.902559e-08,2.973017e-08


MODEL CONFIGURATION / COMPLEXITY


,dataset,encoder,decoder,input_size,MSDC_kernel_sizes,LGAG_kernel_size,expansion_factor,depthwise_parallel,aggregation,activation,batch_size,epochs,learning_rate,runs,FLOPs_G,THOP_params_M,total_parameters_M,checkpoint_missing_keys,checkpoint_unexpected_keys
0,ClinicDB,pvt_v2_b2,EMCAD,352x352,"[1,3,5]",3,2,True,add,relu6,8,200,0.0005,5,10.579868,26.764399,26.764411,[],"['total_ops', 'total_params', 'backbone.total_..."



FINAL OUTPUTS
EMCAD_ClinicDB_5Run_Final_Results.zip 0.73 MB
EMCAD_ClinicDB_5Run_Best_Checkpoints.tar.gz 474.29 MB
EMCAD_ClinicDB_5Run_Final_SHA256SUMS.txt 0.00 MB
EMCAD_ClinicDB_5Run_Replication_Results.xlsx 0.17 MB

FINAL NOTEBOOK COMPLETE
